
### Best Genesys Card Pools
Finding the top-K best card combinations given a tournament budget $\mathcal{B} = 100$.
We first attribuite each card a weight $w(card)$, that being the one assigned by konami, which impacts on the budget $\mathcal{B}$.
Then we also compute a value $V(card)$, which should ideally represent the "goodness" of one card (please see the notes below).

We solve the famous "knapsack problem": our objective is to put inside an imaginary knapsack (let's say Kaiba's briefcase) the best of combinations of cards, the one that maximizes the total value while respecting th imposed budget. It might sound as a faily easy problem to solve... it is not.
In fact, we need something called ILP (Integer Linear Programming) to solve it. I won't stress you with it.

Instead, let's spend few words on the value function $V$. It goes without saying that ***any*** attempt of estimating it comes at the cost of immense bias.
Our own formula is based on the (TCG) banlist which, despite being oftentimes criticized, provides a quantitative measure of a card's power creep. The idea is that:

***the longer a card has been banned, limited or semi-limited, the higher the chance it would have been been exploited by players, who are min-maxing agents.***

The "banned-time" or "limited-time" has to be computed relative to the card's lifespan. Or value function, in first approximation, is given by

$$V(card) \propto \alpha B + \beta L_1 + \gamma L_2 $$

where $\alpha > \beta > \gamma\in\mathbb{R}$ are finetunable coefficents and $B=\frac{\text{banned time}}{\text{life time}}$, $L_1=\frac{\text{limited time}}{\text{life time}}$, $L_2=\frac{\text{semi-limited time}}{\text{life time}}$ (here time = #days). Now, a card can be good or exploitable even if it has never been in the banlist. That's why we need something more. Another easy-to-retrieve datasource is the playing percentage of the card on Master Duel, or the best performance (of the deck it was part of) at a premiere event. While this constitutes another distorted index, it could do its job to assign a some baseline value to almoest each card on the list. Then, our value function $V$ ultimately takes this form

$$V(card) = \alpha B + \beta L_1 + \gamma L_2 + \delta P$$

where $P\in [0,1]$ is a popularity index (BO1 or EVENT, if it the card is treated as STAPLE or ARCHETYPAL rspectively [I did only the ones where there was a pop index in the MDM website cuz laziness]) and $\delta$ is another coefficient, not contstrained like the other ones. I was thinking about also adding a term related to the price trend. Clearly, this just a simplistic attempt and also completely ignores: any card interdependence, any context regarding Yu-Gi-Oh! formats and metagames.

 ***Please take this with a grain of salt: this is made for fun***. 
 
The obtained combinations you see might or might not reflect subjective ideas of "good" or "bad" (or might simply turn out to objectively f suck xD !!!). Your turn to improve this eventually (or "use it properly")!! You might want to reuse this tool by making up your own card pools, regulate the coefficients, change the value fun... the approach stays pretty much the same.

Thank you very much https://yugipedia.com, https://www.timeanddate.com (to compute the # of days) and https://www.masterduelmeta.com for some of the pops.

In [55]:
import pulp # open terminal and >>>pip install pulp if you don't have it
import numpy as np
from collections import defaultdict
from dataclasses import dataclass

In [56]:
@dataclass(frozen=True)
class Item:
    name: str
    weight: float
    life_days: int
    ban_days: int 
    lim1_days: int
    lim2_days: int
    best_performance: float 
    bo1_usage_rate: float 
    max_price: float

Main code block, the function that solves with ILP, the one used to compute $V$ as well as helpers

In [57]:
def compute_value(items, alpha=10, beta=4, gamma=2, delta=10):
    vals = np.zeros(len(items), dtype=float)
    for i in range(len(items)):
        lifelen = items[i].life_days
        if lifelen:
            vals[i] += alpha * (items[i].ban_days / lifelen)  + beta * (items[i].lim1_days / lifelen) + gamma * (items[i].lim2_days / lifelen)
        P = items[i].best_performance or items[i].bo1_usage_rate
        if P:
            vals[i] += delta*P
    return vals

In [58]:
def knapsack_top_k(items, capacity, K = 100, alpha=10, beta=5, gamma=3, delta=10):
    n = len(items)
    values = compute_value(items, alpha=alpha, beta=beta, gamma=gamma, delta=delta)
    prob = pulp.LpProblem("knap", pulp.LpMaximize)

    y = {(i,k): pulp.LpVariable(f"y_{i}_{k}", 0, 1, cat="Binary")
         for i in range(n) for k in range(3)}

    prob += pulp.lpSum(items[i].weight * y[(i,k)] for i in range(n) for k in range(3)) <= capacity
    prob += pulp.lpSum(values[i]      * y[(i,k)] for i in range(n) for k in range(3))

    solver = pulp.PULP_CBC_CMD(msg=False)
    order = [(i,k) for i in range(n) for k in range(3)]
    N = len(order)

    results = []
    for _ in range(K):
        if prob.solve(solver) != 1:
            break

        vec = [int(pulp.value(y[idx]) > 0.5) for idx in order]
        if sum(vec) == 0 and results:
            break

        counts = [sum(vec[3 * i:3 * i + 3]) for i in range(n)]

        total_value  = float(sum(values[i]*counts[i] for i in range(n)))
        total_weight = float(sum(items[i].weight*counts[i] for i in range(n)))

        grouped = []
        for i, c in enumerate(counts):
            if c == 1:
                grouped.append(items[i].name)
            elif c > 1:
                grouped.append(f"{c}x {items[i].name}")

        results.append({
            "total_value": total_value,
            "total_weight": total_weight,
            "items": grouped
        })

        ones  = [y[order[j]] for j in range(N) if vec[j] == 1]
        zeros = [y[order[j]] for j in range(N) if vec[j] == 0]
        prob += pulp.lpSum(ones) + pulp.lpSum(1 - z for z in zeros) <= N - 1

    return results

Genesys list

In [67]:
genesys_items = [
    Item("Abyss Dweller", weight=100, life_days=4702, ban_days=534, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Adamancipator Risen - Dragite", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Agido the Ancient Sentinel", weight=50, life_days=1055, ban_days=631, lim1_days=321, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Albion the Sanctifire Dragon", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Allure of Darkness", weight=5, life_days=6450, ban_days=0, lim1_days=2234, lim2_days=115+366+140, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Amorphactor Pain, the Imagination Dracoverlord", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ancient Gear Advance", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ancient Gear Statue", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("And the Band Played On", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Angel O7", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Anti-Spell Fragrance", weight=100, life_days=7980, ban_days=527, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Appointer of the Red Lotus", weight=50, life_days=5794, ban_days=842, lim1_days=244, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Arcana Force XXI - The World", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Archlord Kristya", weight=100, life_days=5794, ban_days=0, lim1_days=0, lim2_days=1036, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Archnemeses Eschatos", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Archnemeses Protos", weight=100, life_days=1974, ban_days=797, lim1_days=528, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Artifact Scythe", weight=100, life_days=4156, ban_days=955, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Asceticism of the Six Samurai", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ash Blossom & Joyous Spring", weight=15, life_days=3176, ban_days=0, lim1_days=0, lim2_days=730, best_performance=0, bo1_usage_rate=0.877, max_price=0),
    Item("Assault Synchron", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Astral Kuriboh", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Atlantean Dragoons", weight=40, life_days=2729, ban_days=0, lim1_days=682, lim2_days=1039, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Azamina Ilia Silvia", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Azamina Mu Rcielago", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Bahamut Shark", weight=81, life_days=4704, ban_days=172, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Baronne de Fleur", weight=85, life_days=1428, ban_days=528, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.146, max_price=0),
    Item("Barrier of the Voiceless Voice", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Barrier Statue of the Abyss", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Barrier Statue of the Drought", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Barrier Statue of the Heavens", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Barrier Statue of the Inferno", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Barrier Statue of the Stormwinds", weight=60, life_days=6900, ban_days=956, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Barrier Statue of the Torrent", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Beatrice, Lady of the Eternal", weight=100, life_days=3479, ban_days=389, lim1_days=2925, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Big Welcome Labrynth", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Blackwing - Boreastorm the Wicked Wind", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Blackwing - Zephyros the Elite", weight=13, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Blaster, Dragon Ruler of Infernos", weight=7, life_days=4515, ban_days=2986, lim1_days=454+455, lim2_days=66, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Blaze Fenix, the Burning Bombardment Bird", weight=70, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Blazing Cartesia, the Virtuous", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Block Dragon", weight=33, life_days=3339, ban_days=1837, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Bonfire", weight=33, life_days=616, ban_days=0, lim1_days=171, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Book of Eclipse", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Book of Moon", weight=7, life_days=8105, ban_days=0, lim1_days=2771+3246, lim2_days=364, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Brain Research Lab", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Branded Expulsion", weight=33, life_days=1148, ban_days=843, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Branded Fusion", weight=33, life_days=1260, ban_days=0, lim1_days=377, lim2_days=11, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Branded Lost", weight=66, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Brilliant Fusion", weight=33, life_days=3703, ban_days=1904, lim1_days=356+172, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    #Item("Butterfly Dagger - Elma", weight=1, life_days=7969, ban_days=7483, lim1_days=424, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0), clearly an outlier
    Item("Bystial Baldrake", weight=30, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Bystial Dis Pater", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Bystial Druiswurm", weight=30, life_days=1071, ban_days=0, lim1_days=172, lim2_days=0, best_performance=0, bo1_usage_rate=0.225, max_price=0),
    Item("Bystial Magnamhut", weight=33, life_days=1071, ban_days=0, lim1_days=731, lim2_days=0, best_performance=0, bo1_usage_rate=0.286, max_price=0),
    Item("Bystial Saronir", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Called by the Grave", weight=20, life_days=2744, ban_days=0, lim1_days=1838, lim2_days=76, best_performance=0, bo1_usage_rate=0.836, max_price=0),
    Item("Card Destruction", weight=40, life_days=8581, ban_days=1841, lim1_days=3926+2563, lim2_days=207, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Card of Demise", weight=40, life_days=3451, ban_days=0, lim1_days=2075, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Card of Safe Return", weight=40, life_days=8244, ban_days=5868, lim1_days=183, lim2_days=180, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Catapult Turtle", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Centur-Ion Auxila", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Centur-Ion Primera", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Centur-Ion Trudea", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Chain Strike", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Change of Heart", weight=10, life_days=8581, ban_days=6254, lim1_days=1059+1227, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Chaofeng, Phantom of the Yang Zing", weight=13, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Chaos Angel", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Chaos Dragon Levianeer", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Chaos Ruler, the Chaotic Magical Dragon", weight=50, life_days=1876, ban_days=1088, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Chaos Space", weight=40, life_days=1925, ban_days=0, lim1_days=727, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Charge of the Light Brigade", weight=33, life_days=6232, ban_days=0, lim1_days=1856, lim2_days=900, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Chicken Game", weight=7, life_days=3703, ban_days=2925, lim1_days=498, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Cold Wave", weight=100, life_days=8376, ban_days=5322, lim1_days=545, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Confiscation", weight=100, life_days=8410, ban_days=182+6599, lim1_days=912+699, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Contact \"C\"", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Cornfield Coatl", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Cosmic Blazar Dragon", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Creature Swap", weight=1, life_days=8147, ban_days=0, lim1_days=0, lim2_days=1697, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Crimson Dragon", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Crossout Designator", weight=10, life_days=1456, ban_days=0, lim1_days=171, lim2_days=0, best_performance=0, bo1_usage_rate=0.682, max_price=0),
    Item("Crystron Inclusion", weight=33, life_days=245, ban_days=0, lim1_days=10, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Crystron Smiger", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Crystron Sulfador", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Crystron Thystvern", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Cyber Angel Benten", weight=40, life_days=3325, ban_days=0, lim1_days=427, lim2_days=138, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Cyber Dragon Infinity", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Cyber Jar", weight=33, life_days=8411, ban_days=6028, lim1_days=1277+797, lim2_days=118, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Cyber-Stein", weight=27, life_days=7428, ban_days=4531+2434, lim1_days=1588, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("D/D/D Duo-Dawn King Kali Yuga", weight=77, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("D/D/D Wave High King Caesar", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Daigusto Emeral", weight=1, life_days=4746, ban_days=664, lim1_days=2266, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger! Bigfoot!", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger! Chupacabra!", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger! Dogman!", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger! Mothman!", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger! Nessie!", weight=7, life_days=2619, ban_days=0, lim1_days=808, lim2_days=188+937, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger! Ogopogo!", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger! Thunderbird!", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger!? Jackalope?", weight=7, life_days=2535, ban_days=0, lim1_days=1713, lim2_days=260+119, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Danger!? Tsuchinoko?", weight=7, life_days=2535, ban_days=0, lim1_days=1713, lim2_days=260+119, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dark End Evaporation Dragon", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dark Hole", weight=3, life_days=8090, ban_days=401+1613, lim1_days=781+181+1582+726, lim2_days=990, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dark World Archives", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dark World Dealings", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Deception of the Sinful Spoils", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Deck Lockdown", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Deep Sea Aria", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Delinquent Duo", weight=100, life_days=8411, ban_days=218+7300, lim1_days=697+182, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Demise of the Land", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Denglong, First of the Yang Zing", weight=33, life_days=3249, ban_days=2085, lim1_days=454, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Denko Sekka", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Destiny HERO - Destroyer Phoenix Enforcer", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Destiny HERO - Plasma", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Destructive Daruma Karma Cannon", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Diabell, Queen of the White Forest", weight=25, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Diabellstar the Black Witch", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Different Dimension Ground", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dimension Fusion", weight=45, life_days=7879, ban_days=6349, lim1_days=465, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dimension Shifter", weight=10, life_days=2200, ban_days=0, lim1_days=171, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dimensional Barrier", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dinomorphia Intact", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dinomorphia Rexterm", weight=91, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dinowrestler Pankratops", weight=10, life_days=2535, ban_days=0, lim1_days=1441, lim2_days=342, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Divine Arsenal AA-ZEUS - Sky Thunder", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.232, max_price=0),
    Item("Diviner of the Herald", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Djinn Releaser of Rituals", weight=100, life_days=5797, ban_days=3725, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dogmatika Ecclesia, the Virtuous", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Domain of the True Monarchs", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dominus Impulse", weight=30, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.242, max_price=0),
    Item("Dominus Purge", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dominus Spiral", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dracotail Arthalion", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dracotail Faimena", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dracotail Mululu", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dragon Master Magia", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dragonic Diagram", weight=33, life_days=3067, ban_days=0, lim1_days=2064, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dragonmaid Sheou", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dragonmaid Tidying", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dragon's Bind", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Dragon's Light and Darkness", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Droll & Lock Bird", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.482, max_price=0),
    Item("Drytron Alpha Thuban", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Drytron Mu Beta Fafnir", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Duality", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Eclipse Wyvern", weight=33,life_days=4985, ban_days=0, lim1_days=2265, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Effect Veiler", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.281, max_price=0),
    Item("El Shaddoll Apkallone", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("El Shaddoll Winda", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Elder Entity Norden", weight=91, life_days=3662, ban_days=3029, lim1_days=426, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Elder Entity N'tss", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Elemental HERO Stratos", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Elzette, Azamina of the White Forest", weight=22, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Emergency Teleport", weight=40, life_days=6233, ban_days=0, lim1_days=1095+1858, lim2_days=183+139+128, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("EMERGENCY!", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Eva", weight=1, life_days=2808, ban_days=937, lim1_days=128+97, lim2_days=118, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Evenly Matched", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Evilswarm Ouroboros", weight=67, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ext Ryzeal", weight=25, life_days=295, ban_days=0, lim1_days=11, lim2_days=160, best_performance=0, bo1_usage_rate=0.309, max_price=0),
    Item("F.A. Dawn Dragster", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fairy Tail - Snow", weight=85, life_days=3340, ban_days=1105+1089, lim1_days=237, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fiber Jar", weight=30, life_days=8149, ban_days=7484, lim1_days=633, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Final Countdown", weight=100, life_days=7972, ban_days=0, lim1_days=4288, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fire Formation - Tenki", weight=40, life_days=4630, ban_days=0, lim1_days=228, lim2_days=122+188, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fire King Courtier Ulcanix", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fire King High Avatar Kirin", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fishborg Blaster", weight=33, life_days=5876, ban_days=5231, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Floowandereeze & Robina", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Floowandereeze and the Advent of Adventure", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Floowandereeze and the Magnificent Map", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Foolish Burial", weight=33, life_days=6547, ban_days=0, lim1_days=5690, lim2_days=911, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Foolish Burial Goods", weight=3, life_days=3277, ban_days=0, lim1_days=0, lim2_days=1001, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Forbidden Chalice", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Forbidden Droplet", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.173, max_price=0),
    Item("Forbidden Lance", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fossil Dig", weight=40, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fossil Dyna Pachycephalo", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Frightfur Patchwork", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Fusion Destiny", weight=33, life_days=2341, ban_days=0, lim1_days=0, lim2_days=238, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gallant Granite", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gateway of the Six", weight=100, life_days=5799, ban_days=1478, lim1_days=915+2932, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gem-Knight Master Diamond", weight=66, life_days=4748, ban_days=0, lim1_days=2687, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ghost Belle & Haunted Mansion", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ghost Meets Girl - A Masterful Mayakashi Shiranui Saga", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ghost Mourner & Moonlit Chill", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ghost Ogre & Snow Rabbit", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ghost Sister & Spooky Dogwood", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Giant Trunade", weight=40, life_days=8413, ban_days=5142, lim1_days=1645, lim2_days=181, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gigantic Spright", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gimmick Puppet Nightmare", weight=70, life_days=4462, ban_days=294, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Give and Take", weight=91, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gladiator Beast Gyzarus", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gladiator Beast Tamer Editor", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gladiator Proving Ground", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gladiator Rejection", weight=15, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Glow-Up Bulb", weight=21, life_days=5435, ban_days=944+1993, lim1_days=288, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Goblin Biker Grand Entrance", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gold Sarcophagus", weight=10, life_days=2963, ban_days=0, lim1_days=547+103+2435, lim2_days=730, best_performance=0, bo1_usage_rate=0.275, max_price=0),
    Item("Gozen Match", weight=100, life_days=6158, ban_days=0, lim1_days=636, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Graceful Charity", weight=40, life_days=8218, ban_days=239+182+6786, lim1_days=366+183+334, lim2_days=109, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Grapha, Dragon Lord of Dark World", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Grapha, Dragon Overlord of Dark World", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Gruesome Grave Squirmer", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Guardian Chimera", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Guiding Quem, the Virtuous", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Harpie's Feather Duster", weight=30, life_days=8203, ban_days=5856, lim1_days=386+1840, lim2_days=0, best_performance=0, bo1_usage_rate=0.197, max_price=0),
    Item("Harpie's Feather Storm", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Heart of the Blue-Eyes", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Heat Wave", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Heavy Storm", weight=20, life_days=8495, ban_days=465+4410, lim1_days=2674+731, lim2_days=311, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Herald of the Arc Light", weight=50, life_days=3979, ban_days=0, lim1_days=13, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Hot Red Dragon Archfiend King Calamity", weight=1, life_days=3433, ban_days=391, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Hyper Rank-Up-Magic Utopiforce", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ice Ryzeal", weight=20, life_days=297, ban_days=0, lim1_days=0, lim2_days=174, best_performance=0, bo1_usage_rate=0.305, max_price=0),
    Item("Ido the Supreme Magical Force", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Imperial Order", weight=100, life_days=8379, ban_days=4723+1329, lim1_days=633+1774, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Imsety, Glory of Horus", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Incoming Machine!", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Incredible Ecclesia, the Virtuous", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Infernal Flame Banshee", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Infernity Launcher", weight=88, life_days=5624, ban_days=0, lim1_days=5506, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Infinite Impermanence", weight=13, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.769, max_price=0),
    Item("Inspector Boarder", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Instant Fusion", weight=100, life_days=6903, ban_days=0, lim1_days=2006, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Interrupted Kaiju Slumber", weight=33, life_days=3544, ban_days=0, lim1_days=496, lim2_days=170, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Into the Void", weight=3, life_days=5593, ban_days=0, lim1_days=2077, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Invoked Caliga", weight=90, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Jet Synchron", weight=1, life_days=3684, ban_days=609, lim1_days=271, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Jowgen the Spiritualist", weight=100, life_days=8246, ban_days=292, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0), # clear outliar lmao
    Item("Junk Speeder", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("K9-04 Noroi", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("K9-17 \"Ripper\"", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("K9-17 Izuna", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("K9-66a Jokul", weight=13, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("K9-ØØ Lupis", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Kaiser Colosseum", weight=100, life_days=8023, ban_days=3316, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Kashtira Arise-Heart", weight=97, life_days=961, ban_days=733, lim1_days=111, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Kashtira Fenrir", weight=30, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Kashtira Unicorn", weight=30, life_days=1073, ban_days=0, lim1_days=0, lim2_days=209, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Kelbek the Ancient Vanguard", weight=50, life_days=1059, ban_days=635, lim1_days=321, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Keldo the Sacred Protector", weight=1, life_days=1059, ban_days=0, lim1_days=957, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ketu Dracotail", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("King of the Feral Imps", weight=33, life_days=4517, ban_days=12, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("King's Sarcophagus", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Knight Armed Dragon, the Armored Knight Dragon", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Knightmare Corruptor Iblee", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Koa'ki Meiru Drago", weight=75, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Koa'ki Meiru Guardian", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Koa'ki Meiru Overload", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Koa'ki Meiru Sandman", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Koa'ki Meiru Wall", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lady Labrynth of the Silver Castle", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lady's Dragonmaid", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Last Turn", weight=75, life_days=8149, ban_days=7119, lim1_days=0, lim2_days=997, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Last Will", weight=100, life_days=8218, ban_days=6785, lim1_days=333, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lavalval Chain", weight=80, life_days=4747, ban_days=3726, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Left Arm Offering", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Legendary Fire King Ponix", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Legendary Lord Six Samurai - Shi En", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Legendary Six Samurai - Shi En", weight=10, life_days=5345, ban_days=0, lim1_days=942, lim2_days=221, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Level Eater", weight=100, life_days=5798, ban_days=2791, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Life Equalizer", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Light and Darkness Dragonlord", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Light Barrier", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Light End Sublimation Dragon", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lightning Storm", weight=40, life_days=2067, ban_days=0, lim1_days=0, lim2_days=832, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lightsworn Dragonling", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lonefire Blossom", weight=33, life_days=6454, ban_days=0, lim1_days=852, lim2_days=729+454, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lose 1 Turn", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lubellion the Searing Dragon", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lyrilusc - Beryl Canary", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lyrilusc - Bird Call", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Lyrilusc - Independent Nightingale", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Magical Explosion", weight=75, life_days=7362, ban_days=0, lim1_days=5689, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Magical Mid-Breaker Field", weight=60, life_days=3341, ban_days=0, lim1_days=2434, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Magical Scientist", weight=95, life_days=8023, ban_days=7484, lim1_days=423, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Magician of Black Chaos MAX", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Majesty's Fiend", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mansion of the Dreadful Dolls", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Masked HERO Dark Law", weight=70, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mass Driver", weight=100, life_days=8023, ban_days=5324, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Master Peace, the True Dracoslaying King", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mathmech Sigma", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Maxx \"C\"", weight=50, life_days=5345, ban_days=2791, lim1_days=310, lim2_days=213, best_performance=0, bo1_usage_rate=0.955, max_price=0),
    Item("Meizen the Battle Ninja", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mementomictlan Tecuhtlica - Creation King", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mementotlan Bone Party", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mementotlan Twin Dragon", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Metamorphosis", weight=10, life_days=8107, ban_days=6601, lim1_days=515, lim2_days=183, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Metaverse", weight=3, life_days=2900, ban_days=0, lim1_days=1234, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mikanko Water Arabesque", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Millennium Ankh", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mind Drain", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mind Master", weight=1, life_days=6234, ban_days=5140, lim1_days=729, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mirage of Nightmare", weight=10, life_days=8107, ban_days=7485, lim1_days=584, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mirrorjade the Iceblade Dragon", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Miscellaneousaurus", weight=75, life_days=3152, ban_days=0, lim1_days=587+1549, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mistake", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mitsurugi Prayers", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mitsurugi Ritual", weight=60, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Monster Gate", weight=50, life_days=7819, ban_days=0, lim1_days=3968+1328, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Monster Reborn", weight=15, life_days=8456, ban_days=3123, lim1_days=840+548+1095+2791, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Morphing Jar", weight=33, life_days=8397, ban_days=1629, lim1_days=427+3468+2393, lim2_days=241+160, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Morphtronic Telefon", weight=55, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Moulinglacia the Elemental Lord", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mudora the Sword Oracle", weight=1, life_days=1059, ban_days=0, lim1_days=957, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mulcharmy Fuwalos", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.577, max_price=0),
    Item("Mulcharmy Meowls", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mulcharmy Purulia", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.217, max_price=0),
    Item("Multi-Universe", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("M-X-Saber Invoker", weight=33, life_days=5000, ban_days=2567, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Mystic Mine", weight=100, life_days=2340, ban_days=1031, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Nadir Servant", weight=33, life_days=1878, ban_days=0, lim1_days=0, lim2_days=237, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Naturia Barkion", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Naturia Beast", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Naturia Exterio", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Necrovalley", weight=40, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Neptabyss, the Atlantean Prince", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Nibiru, the Primal Being", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.484, max_price=0),
    Item("Nightmare Apprentice", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Nightmare Throne", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 1: Infection Buzzking", weight=85, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 1: Numeron Gate Ekam", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 100: Numeron Dragon", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 16: Shock Master", weight=100, life_days=4868, ban_days=4409, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 2: Numeron Gate Dve", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 3: Numeron Gate Trini", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 38: Hope Harbinger Dragon Titanic Galaxy", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 4: Numeron Gate Catvari", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 40: Gimmick Puppet of Strings", weight=50, life_days=4659, ban_days=0, lim1_days=390, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 41: Bagooska the Terribly Tired Tapir", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.320, max_price=0),
    Item("Number 43: Manipulator of Souls", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 59: Crooked Cook", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 60: Dugares the Timeless", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 67: Pair-a-Dice Smasher", weight=67, life_days=2648, ban_days=12, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 75: Bamboozling Gossip Shadow", weight=70, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 86: Heroic Champion - Rhongomyniad", weight=31, life_days=3817, ban_days=2434, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 89: Diablosis the Mind Hacker", weight=85, life_days=2953, ban_days=845, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 90: Galaxy-Eyes Photon Lord", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 95: Galaxy-Eyes Dark Matter Dragon", weight=50, life_days=3845, ban_days=2343, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number 97: Draglubion", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number C1: Numeron Chaos Gate Sunya", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number C40: Gimmick Puppet of Dark Strings", weight=50, life_days=4202, ban_days=0, lim1_days=390, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number F0: Utopic Draco Future", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Number S0: Utopic ZEXAL", weight=100, life_days=3372, ban_days=1658, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Numbers Eveil", weight=70, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Numeron Calling", weight=30, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Numeron Network", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Obedience Schooled", weight=40, life_days=4265, ban_days=0, lim1_days=12, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ohime the Manifested Mikanko", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ojama Duo", weight=2, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ojama Trio", weight=3, life_days=7971, ban_days=0, lim1_days=913, lim2_days=2713, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("One Day of Peace", weight=11, life_days=5070, ban_days=0, lim1_days=4593, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("One for One", weight=91, life_days=5982, ban_days=0, lim1_days=5870, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Onomatopaira", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Original Sinful Spoils - Snake-Eye", weight=100, life_days=709, ban_days=292, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Outer Entity Azathot", weight=100, life_days=2599, ban_days=2077, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Painful Choice", weight=95, life_days=8412, ban_days=7484, lim1_days=912, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Phantom Knights' Rank-Up-Magic Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Phantom of Yubel", weight=50, life_days=464, ban_days=0, lim1_days=292, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Pilgrim Reaper", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Planet Pathfinder", weight=2, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Pot of Desires", weight=20, life_days=3341, ban_days=0, lim1_days=98, lim2_days=593, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Pot of Greed", weight=30, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Pot of Prosperity", weight=40, life_days=8604, ban_days=7301, lim1_days=1242, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Powersink Stone", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Premature Burial", weight=25, life_days=8378, ban_days=6235, lim1_days=2100, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Preparation of Rites", weight=5, life_days=5798, ban_days=0, lim1_days=900, lim2_days=139, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Pre-Preparation of Rites", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Pressured Planet Wraitsoth", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Primeval Planet Perlereino", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Primite Lordly Lode", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Prohibition", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Pseudo Space", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Psi-Blocker", weight=61, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Psychic End Punisher", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("PSY-Framegear Delta", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("PSY-Framegear Epsilon", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("PSY-Framegear Gamma", weight=15, life_days=3649, ban_days=0, lim1_days=845, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("PSY-Framelord Omega", weight=100, life_days=3649, ban_days=0, lim1_days=2567, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Purrely", weight=15, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Purrely Sleepy Memory", weight=15, life_days=877, ban_days=0, lim1_days=0, lim2_days=635, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Purrelyly", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Quick Launch", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Raidraptor - Vanishing Lanius", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Raigeki", weight=7, life_days=8604, ban_days=3688, lim1_days=109+2685, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic - The Seventh One", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Admiration of the Thousands", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Argent Chaos Force", weight=5, life_days=4202, ban_days=1028, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Astral Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Barian's Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Cipher Ascension", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Doom Double Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Limited Barian's Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Magical Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Numeron Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Quick Chaos", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Raid Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Raptor's Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Revolution Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Skip Force", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Soul Shave Force", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rank-Up-Magic Zexal Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ra's Disciple", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Reasoning", weight=50, life_days=8107, ban_days=0, lim1_days=1460+3456, lim2_days=851, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Red Reboot", weight=50, life_days=2704, ban_days=1090, lim1_days=986, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Red-Eyes Dark Dragoon", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Redox, Dragon Ruler of Boulders", weight=7, life_days=4515, ban_days=2986, lim1_days=454+455, lim2_days=66, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Regenesis", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Reinforcement of the Army", weight=40, life_days=8149, ban_days=0, lim1_days=1960+3610, lim2_days=1881+78, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rescue-ACE Air Lifter", weight=10, life_days=982, ban_days=0, lim1_days=342, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rescue-ACE Preventer", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Return from the Different Dimension", weight=50, life_days=7734, ban_days=4287, lim1_days=1980, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Return of the Dragon Lords", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rise Rank-Up-Magic Raidraptor's Force", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rite of Aramesir", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ritual Beast Tamer Elder", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Rivalry of Warlords", weight=100, life_days=8023, ban_days=0, lim1_days=635, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ronintoadin", weight=60, life_days=5623, ban_days=1090, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Royal Decree", weight=10, life_days=8032, ban_days=0, lim1_days=0, lim2_days=365+183, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Royal Magical Library", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Royal Oppression", weight=100, life_days=8149, ban_days=5140, lim1_days=364, lim2_days=183, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ryzeal Detonator", weight=20, life_days=296, ban_days=0, lim1_days=173, lim2_days=0, best_performance=0, bo1_usage_rate=0.320, max_price=0),
    Item("Ryzeal Duo Drive", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0.305, max_price=0),
    Item("Sales Ban", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Sangen Kaimen", weight=50, life_days=520, ban_days=0, lim1_days=292, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Sangen Summoning", weight=100, life_days=520, ban_days=0, lim1_days=292, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Sauravis, the Ancient and Ascended", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Schwarzschild Infinity Dragon", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Secret Village of the Spellcasters", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Self-Destruct Button", weight=100, life_days=7880, ban_days=4287, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Sengenjin Wakes from a Millennium", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Set Rotation", weight=33, life_days=3068, ban_days=0, lim1_days=2882, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Shaddoll Schism", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Shien's Dojo", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Shien's Smoke Signal", weight=33, life_days=5345, ban_days=0, lim1_days=181, lim2_days=364, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Shooting Riser Dragon", weight=33, life_days=2603, ban_days=0, lim1_days=138, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Sillva, Warlord of Dark World", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Sixth Sense", weight=65, life_days=4370, ban_days=4287, lim1_days=81, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Skill Drain", weight=100, life_days=7971, ban_days=0, lim1_days=2503+390, lim2_days=364, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Smoke Grenade of the Thief", weight=87, life_days=8142, ban_days=3574, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Snake-Eye Ash", weight=5, life_days=923, ban_days=0, lim1_days=216, lim2_days=160, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Snake-Eyes Poplar", weight=5, life_days=597, ban_days=0, lim1_days=216, lim2_days=160, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Snatch Steal", weight=7, life_days=8410, ban_days=180+2678+3196, lim1_days=1430+183+89+635, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Snoww, Unlight of Dark World", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Solemn Judgment", weight=7, life_days=8494, ban_days=1617, lim1_days=1460+524+12, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Solemn Strike", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Solemn Warning", weight=5, life_days=5525, ban_days=0, lim1_days=2515, lim2_days=730, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Songs of the Dominators", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Soul Charge", weight=50, life_days=4174, ban_days=2434, lim1_days=1579, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Soul Drain", weight=100, life_days=4783, ban_days=0, lim1_days=2331, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Speedroid Terrortop", weight=3, life_days=3649, ban_days=0, lim1_days=2393, lim2_days=104, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Spell Canceller", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Spiritual Beast Tamer Lara", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Spright Starter", weight=20, life_days=1150, ban_days=0, lim1_days=0, lim2_days=209, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Stand Up Centur-Ion!", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Star Seraph Scepter", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Star Seraph Sovereignty", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Stardust Sifr Divine Dragon", weight=1, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Starliege Seyfert", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Stray Purrely Street", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Substitoad", weight=60, life_days=6346, ban_days=4414, lim1_days=1090, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Subterror Guru", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Summon Limit", weight=100, life_days=6346, ban_days=530, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Super Polymerization", weight=10, life_days=6454, ban_days=1354, lim1_days=91+223, lim2_days=76, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Super Starslayer TY-PHON - Sky Crisis", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    #Item("Supreme King Dragon Starving Venom", weight=1, life_days=2977, ban_days=2686, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0), outiler (pends)
    Item("Swap Frog", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Sword Ryzeal", weight=20, life_days=296, ban_days=0, lim1_days=0, lim2_days=85, best_performance=0, bo1_usage_rate=0.309, max_price=0),
    Item("Swordsoul Emergence", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Swordsoul Grandmaster - Chixiao", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Swordsoul of Mo Ye", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Swordsoul Strategist Longyuan", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("T.G. Hyper Librarian", weight=33, life_days=5263, ban_days=0, lim1_days=2210+2005, lim2_days=496, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tearlaments Havnis", weight=50, life_days=1150, ban_days=0, lim1_days=957, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tearlaments Kitkallos", weight=50, life_days=1150, ban_days=957, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tearlaments Merrli", weight=50, life_days=1150, ban_days=0, lim1_days=957, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tearlaments Reinoheart", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tearlaments Scheiren", weight=50, life_days=1150, ban_days=0, lim1_days=957, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Telekinetic Charging Cell", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tellarknight Ptolemaeus", weight=100, life_days=3789, ban_days=2487, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tempest, Dragon Ruler of Storms", weight=7, life_days=4515, ban_days=2986, lim1_days=454+455, lim2_days=66, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tenpai Dragon Chundra", weight=50, life_days=520, ban_days=0, lim1_days=292, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tenpai Dragon Genroku", weight=25, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tenyi Spirit - Ashuna", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Terraforming", weight=33, life_days=8107, ban_days=0, lim1_days=2266, lim2_days=360, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("That Grass Looks Greener", weight=50, life_days=2947, ban_days=2295, lim1_days=342+390, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Black Goat Laughs", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Bystial Lubellion", weight=30, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Forceful Sentry", weight=100, life_days=8412, ban_days=7484, lim1_days=912, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Gates of Dark World", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Hidden City", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Last Warrior from Another Planet", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Melody of Awakening Dragon", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Monarchs Erupt", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Phantom Knights' Rank-Up-Magic Launch", weight=1, life_days=3616, ban_days=265, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Tyrant Neptune", weight=100, life_days=5082, ban_days=3102, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Unstoppable Exodia Incarnate", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("The Zombie Vampire", weight=50, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("There Can Be Only One", weight=100, life_days=2795, ban_days=0, lim1_days=635, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Thunder Dragon Colossus", weight=67, life_days=2536, ban_days=1546, lim1_days=140, lim2_days=97, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Thunder King Rai-Oh", weight=20, life_days=6171, ban_days=0, lim1_days=1092, lim2_days=183+524, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tidal, Dragon Ruler of Waterfalls", weight=7, life_days=4515, ban_days=2986, lim1_days=454+455, lim2_days=66, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Toadally Awesome", weight=20, life_days=3250, ban_days=0, lim1_days=165, lim2_days=91, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tour Guide From the Underworld", weight=3, life_days=5259, ban_days=0, lim1_days=1754, lim2_days=486+237, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Transaction Rollback", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Trap Dustshoot", weight=94, life_days=8107, ban_days=4958, lim1_days=1642, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Trap Holic", weight=7, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Traptrix Rafflesia", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Triple Tactics Talent", weight=93, life_days=1878, ban_days=0, lim1_days=173, lim2_days=0, best_performance=0, bo1_usage_rate=0.551, max_price=0),
    Item("Triple Tactics Thrust", weight=13, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Trishula, Dragon of the Ice Barrier", weight=3, life_days=2359, ban_days=1231, lim1_days=181+1459+775, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("True King of All Calamities", weight=100, life_days=3068, ban_days=1657, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Tyrant's Tirade", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Ultimaya Tzolkin", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Union Hangar", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Vanity's Emptiness", weight=100, life_days=5434, ban_days=3102, lim1_days=729, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Vanity's Fiend", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Vanity's Ruler", weight=100, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Vanquish Soul Hollie Sue", weight=10, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Varudras, the Final Bringer of the End Times", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Virtual World Kyubi - Shenshen", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Virtual World Mai-Hime - Lulu", weight=3, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Wandering Gryphon Rider", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("WANTED: Seeker of Sinful Spoils", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Water Enchantress of the Temple", weight=5, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Welcome Labrynth", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Wind-Up Carrier Zenmaity", weight=33, life_days=5000, ban_days=4593, lim1_days=180, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Wind-Up Hunter", weight=75, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Wishes for Eyes of Blue", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Witch of the White Forest", weight=33, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Yaguramaru the Armor Ninja", weight=20, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Zaborg the Mega Monarch", weight=80, life_days=0, ban_days=0, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Zoodiac Barrage", weight=33, life_days=3152, ban_days=1164, lim1_days=547+279, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Zoodiac Broadbull", weight=66, life_days=3152, ban_days=2931, lim1_days=0, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Zoodiac Drident", weight=20, life_days=3152, ban_days=925+1536, lim1_days=455+12, lim2_days=0, best_performance=0, bo1_usage_rate=0, max_price=0),
    Item("Zoodiac Ratpier", weight=50, life_days=3152, ban_days=0, lim1_days=2931, lim2_days=170, best_performance=0, bo1_usage_rate=0, max_price=0),
]

print("count:", len(genesys_items))

count: 534


See the selected cardpool ranked by value

In [65]:
card_pool = genesys_items

alpha = 8; beta = 4; gamma = 2; delta = 10
values = compute_value(card_pool, alpha=alpha, beta=beta, gamma=gamma, delta=delta)
    
pairs = list(zip(card_pool, values))

pairs.sort(key=lambda x: x[1], reverse=True)

for rank, (item, val) in enumerate(pairs, start=1):
    print(f"{rank:>3}. {item.name:<40}  value={val:.4f}")

  1. Maxx "C"                                  value=14.0391
  2. Called by the Grave                       value=11.0947
  3. Ash Blossom & Joyous Spring               value=9.2297
  4. Harpie's Feather Duster                   value=8.7665
  5. Cyber-Stein                               value=8.3565
  6. Trishula, Dragon of the Ice Barrier       value=8.2696
  7. Sixth Sense                               value=7.9222
  8. Infinite Impermanence                     value=7.6900
  9. Mirage of Nightmare                       value=7.6744
 10. Magical Scientist                         value=7.6734
 11. Fiber Jar                                 value=7.6579
 12. Delinquent Duo                            value=7.5687
 13. Painful Choice                            value=7.5511
 14. The Forceful Sentry                       value=7.5511
 15. Wind-Up Carrier Zenmaity                  value=7.4928
 16. Graceful Charity                          value=7.4721
 17. Zoodiac Broadbull                

Get results

In [68]:
if __name__ == "__main__":
    
    tournament_budget = 100

    solutions = knapsack_top_k(card_pool, tournament_budget, alpha=8, beta=4, gamma=2, delta=10) # make sure alpha > beta > gamma if using the above defined V
    
    
    for rank, s in enumerate(solutions, 1):
        items_list = s.get("items") or [f"{cnt}x {name}" if cnt > 1 else name for name, cnt in s.get("solution", {}).items()]
        ll = "; ".join(items_list) if items_list else "(none)"
        print(f"{rank:>3}. value = {s['total_value']:.2f}  weight = {s['total_weight']:.2f}  -> {ll}")

